In [11]:
import numpy as np
import matplotlib.pyplot as plt
import torch 
import torch.nn as nn


#### Implementing CNN Architectures

In [12]:
from functools import partial

In [13]:
device = "cuda" if torch.cuda.is_available() else "cpu"

##### Tackling Fashion Mnist with CNN
- a `28` x `28` image of greyscale channel

In [14]:
torch.manual_seed(42)
default_conv2d = partial(nn.Conv2d, kernel_size=3, padding="same")

model=nn.Sequential(
    default_conv2d(in_channels=1, out_channels=64, kernel_size=7),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    default_conv2d(in_channels=64, out_channels=128),
    nn.ReLU(),
    default_conv2d(in_channels=128, out_channels=128),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    default_conv2d(in_channels=128, out_channels=256),
    nn.ReLU(),
    default_conv2d(in_channels=256, out_channels=256),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Flatten(), 
    nn.Linear(in_features=2304, out_features=128),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(in_features=128, out_features=64),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(in_features=64, out_features=10),    
).to(device=device)

In [15]:
#%pip install torchmetrics

### Using torchmetrics classes
- `metric.update(preds, target)`: Accumulates predictions and targets into internal state for every batch
- `metric.compute()` : Computes final metric from accumulated state at end of batch
- `metric.reset()`: Clears internal state for next epoch
- `metric.forward(preds, target)`: Does update + compute for that batch only

In [16]:
import torchmetrics 

def evaluate_model(model, dataloader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for x_batch, y_batch in dataloader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device) 
            y_pred = model(x_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()




def train_model(model, loss_fn, optimizer, metric, train_loader, valid_loader, epochs):
    history = {"train_loses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(epochs):
        total_loss=0.0
        model.train()
        metric.reset()
        for x_batch, y_batch in train_loader: 
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            y_pred = model(x_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        history["train_loses"].append(total_loss/ len(train_loader))
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(evaluate_model(model=model, dataloader=valid_loader, metric=metric).item())
        
        print( f"Epoch {epoch +1} / {epochs},"
              f"train loss {history['train_loses'][-1]:.4f}, "
              f"train metrics {history['train_metrics'][-1]:.4f}, "
              f"valid metrics {history['valid_metrics'][-1]:.4f}"
        )
    return history
   
    
            
    

In [17]:
import torchvision
import torchvision.transforms.v2 as T

In [18]:
toTensor = T.Compose([
    T.ToImage(), T.ToDtype(torch.float32, scale=True)
])

train_and_valid_dataset = torchvision.datasets.FashionMNIST(
    root="datasets", train=True, transform=toTensor, download=True
)
test_dataset = torchvision.datasets.FashionMNIST(root="datasets", train=False, transform=toTensor, download=True)
torch.manual_seed(42)

train_data , valid_data = torch.utils.data.random_split(
    dataset=train_and_valid_dataset,
    lengths=[55_000, 5_000]
)

In [19]:
from torch.utils.data import DataLoader
torch.manual_seed(42)

train_loader = DataLoader(dataset=train_data, batch_size=32, shuffle=True)
valid_loader = DataLoader(dataset=valid_data, batch_size=32)
test_loader = DataLoader(dataset=test_dataset, batch_size=32)


In [20]:
num_epochs= 25
optimizer = torch.optim.AdamW(model.parameters())
xentropy = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy("multiclass", num_classes=10).to(device)

history = train_model(
                        model=model, 
                        loss_fn=xentropy, 
                        optimizer=optimizer, 
                        metric=accuracy, 
                        train_loader=train_loader,
                        valid_loader=valid_loader,
                        epochs=num_epochs,
                        )



Epoch 1 / 25,train loss 0.9477, train metrics 0.6400, valid metrics 0.7988
Epoch 2 / 25,train loss 0.5583, train metrics 0.7938, valid metrics 0.8330
Epoch 3 / 25,train loss 0.4785, train metrics 0.8311, valid metrics 0.8590
Epoch 4 / 25,train loss 0.4138, train metrics 0.8560, valid metrics 0.8718
Epoch 5 / 25,train loss 0.3811, train metrics 0.8711, valid metrics 0.8850
Epoch 6 / 25,train loss 0.3514, train metrics 0.8790, valid metrics 0.8860
Epoch 7 / 25,train loss 0.3337, train metrics 0.8849, valid metrics 0.8896
Epoch 8 / 25,train loss 0.3142, train metrics 0.8937, valid metrics 0.8822
Epoch 9 / 25,train loss 0.3095, train metrics 0.8949, valid metrics 0.8940
Epoch 10 / 25,train loss 0.2921, train metrics 0.9009, valid metrics 0.8994
Epoch 11 / 25,train loss 0.2822, train metrics 0.9032, valid metrics 0.9030
Epoch 12 / 25,train loss 0.2743, train metrics 0.9060, valid metrics 0.9028
Epoch 13 / 25,train loss 0.2642, train metrics 0.9102, valid metrics 0.8996
Epoch 14 / 25,train l